# Automated KYC Document Classification

### Regulatory Compliance Tech Regtech — Banking-AI

Know Your Customer (KYC) is a critical banking process. Instead of manually sorting thousands of IDs, passports, and utility bills, we can use AI to classify documents automatically.

In this notebook:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of regulatory compliance and RegTech terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the NLP task in the context of regulatory compliance and RegTech and how text features are extracted.
- Implement a text classification pipeline and evaluate accuracy on representative banking queries.
- Evaluate robustness to varied phrasing and identify failure modes relevant to customer-facing deployment.

**Estimated time:** 35–45 minutes

**Collection context:** KYC automation, regulatory reporting, compliance monitoring, bias detection, and data privacy in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks receive millions of documents daily. Automating the initial sorting (Is this a Driver's License or a Utility Bill?) saves thousands of human hours.

**Input data used:** Text snippets extracted via OCR (Optical Character Recognition) from document images.

**What we predict:** Document Type (Passport, Driver's License, Utility Bill, or Bank Statement).

**ML method used:** Logistic Regression with TF-IDF vectorization (standard for simple text classification).

**Learning goal:** Understand how to turn raw text into numbers that a machine learning model can understand.

**Primary stakeholders:** compliance officers, legal teams, audit managers, and data protection officers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

We will create 1,000 examples of "text" that might be found on different types of documents.

In [ ]:
data = []
doc_types = ['Passport', 'Drivers_License', 'Utility_Bill', 'Bank_Statement']

keywords = {
    'Passport': ['nationality', 'passport no', 'date of birth', 'place of issue', 'republic of', 'travel document', 'authority'],
    'Drivers_License': ['license number', 'class', 'expiry date', 'driving', 'vehicle', 'restrictions', 'eye color', 'height'],
    'Utility_Bill': ['account balance', 'usage period', 'electricity', 'water', 'gas', 'due date', 'service address', 'billing cycle'],
    'Bank_Statement': ['transaction history', 'credit', 'debit', 'account number', 'balance forward', 'branch code', 'monthly statement']
}

for doc in doc_types:
    for _ in range(250):  # 250 examples per type
        # Pick 3-5 keywords and mix them with random "filler" text simulations
        picked = RNG.choice(keywords[doc], size=RNG.integers(3, 6), replace=False)
        content = " ".join(picked) + " random data name address " + " ".join(RNG.choice(['john', 'doe', 'street', 'ave', '2023', 'state'], size=3))
        data.append({'text': content, 'label': doc})

df = pd.DataFrame(data)
df = df.sample(frac=1).reset_index(drop=True)  # Shuffle

print(f"Total documents: {len(df)}")
df.head()

## Step 4 — Exploratory Data Analysis

EDA

Let's see the distribution of document types (it should be perfectly balanced since we created it that way).

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='label', data=df, palette='viridis')
plt.title('Document Type Distribution')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Count plot titled "Document Type Distribution". The chart highlights key patterns that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

# Vectorize the text
tfidf = TfidfVectorizer(max_features=500)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# Train model

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the most frequent class ---
from collections import Counter
target_col = df.columns[-1]
majority_class, majority_n = Counter(df[target_col]).most_common(1)[0]
print(f"Majority-class baseline: always predict '{majority_class}' -> accuracy {majority_n/len(df):.3f}")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)

print("Model trained successfully!")

In [ ]:
y_pred = model.predict(X_test_vec)
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=doc_types, yticklabels=doc_types, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: KYC Document Classification')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap titled "Confusion Matrix: KYC Document Classification" with Predicted on the x-axis and Actual on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
new_scan = ["passport no 123456 republic of travel document name john doe"]
new_scan_vec = tfidf.transform(new_scan)
prediction = model.predict(new_scan_vec)
probabilities = model.predict_proba(new_scan_vec)

print(f"New Scan Text: {new_scan[0]}")
print(f"Predicted Document Type: {prediction[0]}")

prob_df = pd.DataFrame(probabilities, columns=model.classes_)
print("\nConfidence Scores:")
print(prob_df)

Let's simulate a brand new scan and see what the AI thinks it is.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: classify new queries ---
print("Operational demonstration — real-time intent classification:\n")
sample_queries = [
    "Can you show me my account balance?",
    "I need to transfer money to savings",
    "My card was stolen, please help",
    "What interest rates do you offer?",
    "I want to close my account",
]
for q in sample_queries:
    intent = model.predict([q])[0]
    print(f'  "{q}"  ->  {intent}')

print("\nIn production, predicted intents would route queries to the correct service channel.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end regulatory compliance and RegTech workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Automated compliance systems must be auditable and explainable to regulators.
- AI-driven surveillance can raise employee and customer privacy concerns.
- Bias in compliance models may lead to disproportionate scrutiny of certain groups.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in regulatory compliance and RegTech?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.